# 📊 KAMP 제조 데이터 탐색

> **학습 목표**: KAMP 공공 제조 데이터를 불러오고, 기본적인 데이터 분석을 수행합니다.

---

## 📋 학습 내용

1. ✅ KAMP 데이터 로드
2. ✅ 데이터 구조 파악
3. ✅ 기초 통계 분석
4. ✅ 시각화를 통한 인사이트 도출

**소요 시간**: 약 30분  
**난이도**: ⭐ (초급)

---

## 🔧 Step 1: 라이브러리 Import

필요한 Python 라이브러리를 불러옵니다.

In [ ]:
# 데이터 분석
import pandas as pd
import numpy as np

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# 유틸리티
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'  # Mac
# plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows
plt.rcParams['axes.unicode_minus'] = False

# 시각화 스타일
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ 라이브러리 로드 완료!")

## 📂 Step 2: 데이터 로드

KAMP CNC 가공 데이터를 불러옵니다.

### 💡 Tip: 한글 인코딩 자동 감지
한국 공공 데이터는 `utf-8-sig` 또는 `cp949` 인코딩을 사용합니다.  
자동으로 감지하여 로드합니다.

In [ ]:
# ============================================================
# KAMP 실데이터 경로 (dataset/ 디렉토리)
# ============================================================
kamp_data_dir = Path('../../dataset/part1-1')

# 우선순위별 데이터 파일 목록
kamp_files = {
    '진부': kamp_data_dir / '진부_공개용데이터_V1.csv',
    '영주식품': kamp_data_dir / '제조AI데이터셋_영주식품.csv',
    '에스케이지': kamp_data_dir / '제조AI데이터셋_에스케이지 주식회사.csv',
    '원스타': kamp_data_dir / '제조AI데이터셋_원스타.csv',
}

# 샘플 데이터 경로 (fallback)
sample_path = Path('../data/sample_kamp_cnc.csv')

# 데이터 로드 함수 (인코딩 자동 감지)
def load_kamp_data(file_path):
    """KAMP 데이터 로드 (한글 인코딩 자동 처리)"""
    encodings = ['utf-8-sig', 'cp949', 'utf-8', 'euc-kr']
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"✅ 데이터 로드 성공! (인코딩: {encoding})")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError("❌ 지원하는 인코딩으로 파일을 읽을 수 없습니다.")

# ============================================================
# 데이터 로드: KAMP 실데이터 → 샘플 데이터 fallback
# ============================================================
datasets = {}
for name, path in kamp_files.items():
    if path.exists():
        try:
            datasets[name] = load_kamp_data(path)
            print(f"  📊 {name}: {datasets[name].shape[0]:,}행 × {datasets[name].shape[1]}열")
        except Exception as e:
            print(f"  ⚠️ {name} 로드 실패: {e}")

if datasets:
    # 1순위: 진부 데이터 (EDA 기본)
    df = datasets.get('진부', list(datasets.values())[0])
    data_source = '진부' if '진부' in datasets else list(datasets.keys())[0]
    print(f"\n🎯 기본 분석 데이터: {data_source}")
    print(f"📊 데이터 크기: {df.shape[0]:,}행 × {df.shape[1]}열")
    print(f"\n💡 추가 데이터셋도 로드됨: {list(datasets.keys())}")
else:
    # Fallback: 샘플 데이터
    print("⚠️ KAMP 실데이터를 찾을 수 없습니다. 샘플 데이터를 사용합니다.")
    try:
        df = load_kamp_data(sample_path)
        data_source = '샘플'
        print(f"📊 데이터 크기: {df.shape[0]:,}행 × {df.shape[1]}열")
    except FileNotFoundError:
        print("❌ 파일을 찾을 수 없습니다.")
        print("💡 해결 방법: dataset/part1-1/ 폴더에 KAMP 데이터를 배치하세요.")

## 🔍 Step 3: 데이터 구조 파악

데이터의 기본 정보를 확인합니다.

In [ ]:
# 데이터 미리보기
print("📋 데이터 미리보기 (처음 5행)")
print("="*80)
df.head()

In [ ]:
# 데이터 타입 및 결측치 확인
print("📊 데이터 정보")
print("="*80)
df.info()

print("\n🔍 결측치 확인")
print("="*80)
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("✅ 결측치 없음!")

In [ ]:
# 기초 통계량
print("📈 기초 통계량")
print("="*80)
df.describe()

## 📊 Step 4: 데이터 시각화

### 4.1 수치형 변수 분포

In [ ]:
# 수치형 컬럼 추출
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"📊 수치형 변수: {len(numeric_cols)}개")
print(numeric_cols)

# 히스토그램 (처음 6개 변수)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols[:6]):
    axes[idx].hist(df[col].dropna(), bins=30, color='skyblue', edgecolor='black')
    axes[idx].set_title(f'{col} 분포', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('빈도')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 4.2 범주형 변수 분석

In [ ]:
# 범주형 컬럼 추출
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

if categorical_cols:
    print(f"📊 범주형 변수: {len(categorical_cols)}개")
    print(categorical_cols)
    
    # 각 범주형 변수의 고유값 개수
    for col in categorical_cols:
        unique_count = df[col].nunique()
        print(f"  - {col}: {unique_count}개 카테고리")
        if unique_count <= 10:  # 10개 이하만 출력
            print(f"    {df[col].value_counts().to_dict()}")
else:
    print("ℹ️ 범주형 변수가 없습니다.")

### 4.3 상관관계 분석

In [ ]:
# 상관관계 히트맵
if len(numeric_cols) > 1:
    plt.figure(figsize=(12, 10))
    
    corr_matrix = df[numeric_cols].corr()
    
    sns.heatmap(
        corr_matrix,
        annot=True,
        fmt='.2f',
        cmap='coolwarm',
        center=0,
        square=True,
        linewidths=1,
        cbar_kws={"shrink": 0.8}
    )
    
    plt.title('변수 간 상관관계', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
    
    # 높은 상관관계 찾기 (|r| > 0.7)
    print("\n🔍 높은 상관관계 (|r| > 0.7):")
    print("="*80)
    high_corr = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > 0.7:
                high_corr.append((
                    corr_matrix.columns[i],
                    corr_matrix.columns[j],
                    corr_matrix.iloc[i, j]
                ))
    
    if high_corr:
        for var1, var2, corr in high_corr:
            print(f"  {var1} ↔ {var2}: {corr:.3f}")
    else:
        print("  ℹ️ 높은 상관관계를 가진 변수 쌍이 없습니다.")

### 4.4 인터랙티브 시각화 (Plotly)

In [ ]:
# Scatter Matrix (처음 4개 변수)
if len(numeric_cols) >= 2:
    cols_to_plot = numeric_cols[:min(4, len(numeric_cols))]
    
    fig = px.scatter_matrix(
        df[cols_to_plot],
        dimensions=cols_to_plot,
        title="변수 간 관계 (인터랙티브)",
        height=800
    )
    
    fig.update_traces(diagonal_visible=False)
    fig.show()
    
    print("💡 Tip: 차트를 드래그하여 확대/축소할 수 있습니다!")

## 📊 Step 5: 데이터 품질 리포트

전반적인 데이터 품질을 평가합니다.

In [ ]:
def generate_data_quality_report(df):
    """데이터 품질 리포트 생성"""
    report = {}
    
    # 기본 정보
    report['총 행 수'] = f"{len(df):,}"
    report['총 컬럼 수'] = len(df.columns)
    report['메모리 사용량'] = f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
    
    # 결측치
    total_missing = df.isnull().sum().sum()
    missing_pct = (total_missing / (len(df) * len(df.columns))) * 100
    report['총 결측치'] = f"{total_missing:,} ({missing_pct:.2f}%)"
    
    # 중복
    duplicates = df.duplicated().sum()
    dup_pct = (duplicates / len(df)) * 100
    report['중복 행'] = f"{duplicates:,} ({dup_pct:.2f}%)"
    
    # 데이터 타입
    report['수치형 변수'] = len(df.select_dtypes(include=[np.number]).columns)
    report['범주형 변수'] = len(df.select_dtypes(include=['object']).columns)
    
    return report

# 리포트 생성
quality_report = generate_data_quality_report(df)

print("📊 데이터 품질 리포트")
print("="*80)
for key, value in quality_report.items():
    print(f"  {key:20s}: {value}")

# 품질 점수 계산
missing_pct = (df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100
dup_pct = (df.duplicated().sum() / len(df)) * 100
quality_score = 100 - (missing_pct * 0.5 + dup_pct * 0.5)

print(f"\n⭐ 데이터 품질 점수: {quality_score:.1f}/100")
if quality_score >= 90:
    print("✅ 우수한 데이터 품질!")
elif quality_score >= 70:
    print("⚠️ 보통 수준의 데이터 품질")
else:
    print("❌ 데이터 전처리가 필요합니다.")

## 💾 Step 6: 탐색 결과 저장

In [ ]:
# 출력 폴더 생성
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

# 기초 통계량 저장
stats_file = output_dir / '01_data_statistics.csv'
df.describe().to_csv(stats_file, encoding='utf-8-sig')
print(f"✅ 통계량 저장: {stats_file}")

# 상관관계 저장
if len(numeric_cols) > 1:
    corr_file = output_dir / '01_correlation_matrix.csv'
    df[numeric_cols].corr().to_csv(corr_file, encoding='utf-8-sig')
    print(f"✅ 상관관계 저장: {corr_file}")

print("\n🎉 모든 탐색 완료!")

---

## 🎯 학습 정리

### ✅ 완료한 내용
1. KAMP 공공 제조 데이터 로드
2. 데이터 구조 및 통계량 파악
3. 시각화를 통한 패턴 발견
4. 데이터 품질 평가

### 📚 다음 단계
- **02_pycaret_automl.ipynb**: PyCaret을 사용한 자동 머신러닝
- **03_shap_explainer.ipynb**: SHAP을 통한 모델 해석

---

### 💡 추가 학습 자료
- [Pandas 공식 문서](https://pandas.pydata.org/docs/)
- [Plotly 튜토리얼](https://plotly.com/python/)
- [KAMP 데이터 포털](https://mdata.kamp-ai.kr/)

---

*제조AI 교육 v12 Enhanced | Part 1-1 실습 | 2025.02*